# 00 — Common GPT Components

## Scientific Abstract GPT

This notebook creates a reusable Python module containing the common model
architecture and generation utilities used by the remaining notebooks.

The generated module is:

`src/gpt_components.py`

### Common components

- GPT configuration
- Random seed configuration
- Token embeddings
- Positional embeddings
- Masked self-attention
- Multi-head attention
- Feed-forward network
- Transformer block
- GPT language model
- Tokenizer loading
- Prompt creation
- Text generation
- Abstract extraction
- Model checkpoint loading
- Parameter counting

### Notebooks using this module

- `03_GPT_Model.ipynb`
- `04_Training.ipynb`
- `05_Text_Generation.ipynb`
- `06_Evaluation.ipynb`
- `07_Demo.ipynb`

In [1]:
!pip install -q tokenizers

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import sys
import json
import importlib

In [4]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

SRC_FOLDER = os.path.join(
    PROJECT_PATH,
    "src"
)

COMMON_MODULE_PATH = os.path.join(
    SRC_FOLDER,
    "gpt_components.py"
)

INIT_FILE_PATH = os.path.join(
    SRC_FOLDER,
    "__init__.py"
)

os.makedirs(
    SRC_FOLDER,
    exist_ok=True
)

print("Project path:")
print(PROJECT_PATH)

print("\nSource folder:")
print(SRC_FOLDER)

print("\nCommon module path:")
print(COMMON_MODULE_PATH)

Project path:
/content/drive/MyDrive/Scientific-Abstract-GPT

Source folder:
/content/drive/MyDrive/Scientific-Abstract-GPT/src

Common module path:
/content/drive/MyDrive/Scientific-Abstract-GPT/src/gpt_components.py


Create src/__init__.py

In [5]:
init_file_content = '''"""
Reusable source modules for Scientific Abstract GPT.
"""

from .gpt_components import (
    GPTConfig,
    CausalSelfAttentionHead,
    MultiHeadAttention,
    FeedForward,
    TransformerBlock,
    GPTLanguageModel,
    set_seed,
    count_parameters,
    load_tokenizer,
    create_prompt,
    generate_text,
    extract_abstract,
    load_model_checkpoint
)

__all__ = [
    "GPTConfig",
    "CausalSelfAttentionHead",
    "MultiHeadAttention",
    "FeedForward",
    "TransformerBlock",
    "GPTLanguageModel",
    "set_seed",
    "count_parameters",
    "load_tokenizer",
    "create_prompt",
    "generate_text",
    "extract_abstract",
    "load_model_checkpoint"
]
'''

with open(
    INIT_FILE_PATH,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        init_file_content
    )

print(
    "__init__.py created successfully:"
)

print(
    INIT_FILE_PATH
)

__init__.py created successfully:
/content/drive/MyDrive/Scientific-Abstract-GPT/src/__init__.py


Create src/gpt_components.py

In [6]:
common_module_code = r'''
"""
Common GPT components for the Scientific Abstract GPT project.

This module contains:

1. GPT configuration
2. Reproducibility utilities
3. Masked self-attention
4. Multi-head attention
5. Feed-forward network
6. Transformer blocks
7. Decoder-only GPT language model
8. Tokenizer loading
9. Prompt creation
10. Autoregressive text generation
11. Abstract extraction
12. Model checkpoint loading
13. Parameter counting
"""

from __future__ import annotations

import random
from dataclasses import asdict, dataclass
from typing import Any, Dict, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer


# ============================================================
# 1. Reproducibility
# ============================================================

def set_seed(
    seed: int = 42,
    deterministic: bool = False
) -> None:
    """
    Set random seeds for Python, NumPy, and PyTorch.

    Parameters
    ----------
    seed:
        Random seed value.

    deterministic:
        When True, PyTorch attempts to use deterministic
        algorithms. This can reduce performance and may not
        be supported by every GPU operation.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        try:
            torch.use_deterministic_algorithms(
                True
            )
        except Exception:
            pass

    else:

        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# ============================================================
# 2. GPT Configuration
# ============================================================

@dataclass
class GPTConfig:
    """
    Configuration for the decoder-only GPT model.
    """

    vocab_size: int
    block_size: int = 256
    n_embd: int = 256
    n_head: int = 8
    n_layer: int = 6
    dropout: float = 0.1
    bias: bool = True

    def __post_init__(self) -> None:
        """
        Validate configuration values.
        """

        if self.vocab_size <= 0:

            raise ValueError(
                "vocab_size must be greater than zero."
            )

        if self.block_size <= 0:

            raise ValueError(
                "block_size must be greater than zero."
            )

        if self.n_embd <= 0:

            raise ValueError(
                "n_embd must be greater than zero."
            )

        if self.n_head <= 0:

            raise ValueError(
                "n_head must be greater than zero."
            )

        if self.n_layer <= 0:

            raise ValueError(
                "n_layer must be greater than zero."
            )

        if self.n_embd % self.n_head != 0:

            raise ValueError(
                "n_embd must be divisible by n_head. "
                f"Received n_embd={self.n_embd} and "
                f"n_head={self.n_head}."
            )

        if not 0.0 <= self.dropout < 1.0:

            raise ValueError(
                "dropout must be between 0 and 1."
            )

    @property
    def head_size(self) -> int:
        """
        Embedding dimension handled by one attention head.
        """

        return self.n_embd // self.n_head

    def to_dict(self) -> Dict[str, Any]:
        """
        Convert configuration to a dictionary.
        """

        return asdict(self)

    @classmethod
    def from_dict(
        cls,
        config_dictionary: Dict[str, Any]
    ) -> "GPTConfig":
        """
        Create GPTConfig from a dictionary.

        Extra checkpoint keys are ignored.
        """

        valid_keys = {
            "vocab_size",
            "block_size",
            "n_embd",
            "n_head",
            "n_layer",
            "dropout",
            "bias"
        }

        filtered_config = {
            key: value
            for key, value
            in config_dictionary.items()
            if key in valid_keys
        }

        return cls(
            **filtered_config
        )


# ============================================================
# 3. Single Masked Self-Attention Head
# ============================================================

class CausalSelfAttentionHead(nn.Module):
    """
    One masked self-attention head.

    The causal mask ensures that a token can attend only to
    itself and previous tokens, not future tokens.
    """

    def __init__(
        self,
        config: GPTConfig,
        head_size: Optional[int] = None
    ) -> None:

        super().__init__()

        self.config = config

        self.head_size = (
            head_size
            if head_size is not None
            else config.head_size
        )

        self.key = nn.Linear(
            config.n_embd,
            self.head_size,
            bias=config.bias
        )

        self.query = nn.Linear(
            config.n_embd,
            self.head_size,
            bias=config.bias
        )

        self.value = nn.Linear(
            config.n_embd,
            self.head_size,
            bias=config.bias
        )

        self.attention_dropout = nn.Dropout(
            config.dropout
        )

        causal_mask = torch.tril(
            torch.ones(
                config.block_size,
                config.block_size,
                dtype=torch.bool
            )
        )

        self.register_buffer(
            "causal_mask",
            causal_mask,
            persistent=False
        )

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:
        """
        Apply masked scaled dot-product attention.

        Parameters
        ----------
        x:
            Tensor with shape:
            [batch_size, sequence_length, n_embd]

        Returns
        -------
        Tensor with shape:
        [batch_size, sequence_length, head_size]
        """

        batch_size, sequence_length, _ = x.shape

        if sequence_length > self.config.block_size:

            raise ValueError(
                "Input sequence length exceeds block_size. "
                f"Received {sequence_length}, but block_size "
                f"is {self.config.block_size}."
            )

        keys = self.key(x)
        queries = self.query(x)
        values = self.value(x)

        attention_scores = (
            queries @ keys.transpose(-2, -1)
        )

        attention_scores = (
            attention_scores /
            (self.head_size ** 0.5)
        )

        mask = self.causal_mask[
            :sequence_length,
            :sequence_length
        ]

        attention_scores = (
            attention_scores.masked_fill(
                ~mask,
                float("-inf")
            )
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = (
            self.attention_dropout(
                attention_weights
            )
        )

        output = (
            attention_weights @ values
        )

        return output


# ============================================================
# 4. Multi-Head Masked Self-Attention
# ============================================================

class MultiHeadAttention(nn.Module):
    """
    Multiple masked self-attention heads operating in parallel.
    """

    def __init__(
        self,
        config: GPTConfig
    ) -> None:

        super().__init__()

        self.config = config

        self.heads = nn.ModuleList([
            CausalSelfAttentionHead(
                config=config,
                head_size=config.head_size
            )
            for _ in range(
                config.n_head
            )
        ])

        self.output_projection = nn.Linear(
            config.n_embd,
            config.n_embd,
            bias=config.bias
        )

        self.residual_dropout = nn.Dropout(
            config.dropout
        )

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:
        """
        Apply all attention heads and combine their outputs.
        """

        multi_head_output = torch.cat(
            [
                head(x)
                for head in self.heads
            ],
            dim=-1
        )

        projected_output = (
            self.output_projection(
                multi_head_output
            )
        )

        return self.residual_dropout(
            projected_output
        )


# ============================================================
# 5. Feed-Forward Network
# ============================================================

class FeedForward(nn.Module):
    """
    Position-wise feed-forward neural network.

    GPT-style feed-forward networks expand the embedding
    dimension by a factor of four and then project it back.
    """

    def __init__(
        self,
        config: GPTConfig
    ) -> None:

        super().__init__()

        hidden_size = (
            4 * config.n_embd
        )

        self.network = nn.Sequential(
            nn.Linear(
                config.n_embd,
                hidden_size,
                bias=config.bias
            ),
            nn.GELU(),
            nn.Linear(
                hidden_size,
                config.n_embd,
                bias=config.bias
            ),
            nn.Dropout(
                config.dropout
            )
        )

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        return self.network(x)


# ============================================================
# 6. Transformer Block
# ============================================================

class TransformerBlock(nn.Module):
    """
    One decoder-only Transformer block.

    The block contains:

    1. Layer normalization
    2. Masked multi-head self-attention
    3. Residual connection
    4. Layer normalization
    5. Feed-forward network
    6. Residual connection
    """

    def __init__(
        self,
        config: GPTConfig
    ) -> None:

        super().__init__()

        self.layer_norm_attention = nn.LayerNorm(
            config.n_embd
        )

        self.self_attention = (
            MultiHeadAttention(
                config
            )
        )

        self.layer_norm_feed_forward = (
            nn.LayerNorm(
                config.n_embd
            )
        )

        self.feed_forward = (
            FeedForward(
                config
            )
        )

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        x = (
            x +
            self.self_attention(
                self.layer_norm_attention(
                    x
                )
            )
        )

        x = (
            x +
            self.feed_forward(
                self.layer_norm_feed_forward(
                    x
                )
            )
        )

        return x


# ============================================================
# 7. GPT Language Model
# ============================================================

class GPTLanguageModel(nn.Module):
    """
    Decoder-only GPT language model.

    The model predicts the next token for each position in
    the input sequence.
    """

    def __init__(
        self,
        config: GPTConfig
    ) -> None:

        super().__init__()

        self.config = config

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.n_embd
        )

        self.position_embedding = nn.Embedding(
            config.block_size,
            config.n_embd
        )

        self.embedding_dropout = nn.Dropout(
            config.dropout
        )

        self.transformer_blocks = nn.Sequential(
            *[
                TransformerBlock(
                    config
                )
                for _ in range(
                    config.n_layer
                )
            ]
        )

        self.final_layer_norm = nn.LayerNorm(
            config.n_embd
        )

        self.language_model_head = nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False
        )

        # Weight tying:
        # The input token embedding and output token
        # projection share the same weight matrix.
        self.language_model_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._initialize_weights
        )

        self._initialize_residual_projections()

    def _initialize_weights(
        self,
        module: nn.Module
    ) -> None:
        """
        Initialize model weights.
        """

        if isinstance(
            module,
            nn.Linear
        ):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:

                torch.nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding
        ):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def _initialize_residual_projections(
        self
    ) -> None:
        """
        Apply scaled initialization to residual projections.

        This improves stability in deeper Transformer models.
        """

        residual_scale = (
            0.02 /
            ((2 * self.config.n_layer) ** 0.5)
        )

        for module_name, parameter in (
            self.named_parameters()
        ):

            if (
                module_name.endswith(
                    "output_projection.weight"
                )
                or module_name.endswith(
                    "network.2.weight"
                )
            ):

                torch.nn.init.normal_(
                    parameter,
                    mean=0.0,
                    std=residual_scale
                )

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[
            torch.Tensor
        ] = None
    ) -> Tuple[
        torch.Tensor,
        Optional[torch.Tensor]
    ]:
        """
        Perform a forward pass.

        Parameters
        ----------
        input_ids:
            Token IDs with shape:
            [batch_size, sequence_length]

        targets:
            Next-token target IDs with shape:
            [batch_size, sequence_length]

        Returns
        -------
        logits:
            Prediction scores with shape:
            [batch_size, sequence_length, vocab_size]

        loss:
            Cross-entropy loss when targets are provided.
            Otherwise, None.
        """

        if input_ids.ndim != 2:

            raise ValueError(
                "input_ids must have shape "
                "[batch_size, sequence_length]."
            )

        batch_size, sequence_length = (
            input_ids.shape
        )

        if sequence_length > self.config.block_size:

            raise ValueError(
                "Input sequence length exceeds block_size. "
                f"Received {sequence_length}, but block_size "
                f"is {self.config.block_size}."
            )

        positions = torch.arange(
            start=0,
            end=sequence_length,
            device=input_ids.device,
            dtype=torch.long
        )

        token_embeddings = (
            self.token_embedding(
                input_ids
            )
        )

        position_embeddings = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_embeddings +
            position_embeddings
        )

        x = self.embedding_dropout(x)

        x = self.transformer_blocks(x)

        x = self.final_layer_norm(x)

        logits = self.language_model_head(
            x
        )

        loss = None

        if targets is not None:

            if targets.shape != input_ids.shape:

                raise ValueError(
                    "targets must have the same shape "
                    "as input_ids."
                )

            flattened_logits = logits.reshape(
                batch_size * sequence_length,
                self.config.vocab_size
            )

            flattened_targets = targets.reshape(
                batch_size * sequence_length
            )

            loss = F.cross_entropy(
                flattened_logits,
                flattened_targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 200,
        temperature: float = 0.8,
        top_k: Optional[int] = 40,
        top_p: Optional[float] = None,
        repetition_penalty: float = 1.0,
        end_token_id: Optional[int] = None
    ) -> torch.Tensor:
        """
        Generate tokens autoregressively.

        Parameters
        ----------
        input_ids:
            Initial prompt token IDs with shape
            [batch_size, prompt_length].

        max_new_tokens:
            Maximum number of tokens to generate.

        temperature:
            Sampling temperature. Lower values make generation
            more deterministic.

        top_k:
            Keep only the top-k most probable tokens.
            Set to None or 0 to disable.

        top_p:
            Nucleus-sampling probability threshold.
            Set to None to disable.

        repetition_penalty:
            Values greater than 1 reduce repeated-token
            probabilities.

        end_token_id:
            Stop generation when this token is produced.
        """

        if input_ids.ndim != 2:

            raise ValueError(
                "input_ids must have shape "
                "[batch_size, sequence_length]."
            )

        if input_ids.size(1) == 0:

            raise ValueError(
                "The generation prompt cannot be empty."
            )

        if max_new_tokens < 0:

            raise ValueError(
                "max_new_tokens cannot be negative."
            )

        if temperature <= 0:

            raise ValueError(
                "temperature must be greater than zero."
            )

        if repetition_penalty < 1.0:

            raise ValueError(
                "repetition_penalty must be at least 1.0."
            )

        if (
            top_p is not None
            and not 0.0 < top_p <= 1.0
        ):

            raise ValueError(
                "top_p must be between 0 and 1."
            )

        generated_ids = input_ids

        for _ in range(
            max_new_tokens
        ):

            model_context = generated_ids[
                :,
                -self.config.block_size:
            ]

            logits, _ = self(
                model_context
            )

            next_token_logits = logits[
                :,
                -1,
                :
            ].clone()

            if repetition_penalty > 1.0:

                for batch_index in range(
                    generated_ids.size(0)
                ):

                    previous_tokens = torch.unique(
                        generated_ids[
                            batch_index
                        ]
                    )

                    previous_logits = (
                        next_token_logits[
                            batch_index,
                            previous_tokens
                        ]
                    )

                    penalized_logits = torch.where(
                        previous_logits < 0,
                        previous_logits *
                        repetition_penalty,
                        previous_logits /
                        repetition_penalty
                    )

                    next_token_logits[
                        batch_index,
                        previous_tokens
                    ] = penalized_logits

            next_token_logits = (
                next_token_logits /
                temperature
            )

            if (
                top_k is not None
                and top_k > 0
            ):

                effective_top_k = min(
                    int(top_k),
                    next_token_logits.size(-1)
                )

                top_values, _ = torch.topk(
                    next_token_logits,
                    effective_top_k,
                    dim=-1
                )

                top_k_cutoff = top_values[
                    :,
                    -1
                ].unsqueeze(-1)

                next_token_logits = (
                    next_token_logits.masked_fill(
                        next_token_logits
                        < top_k_cutoff,
                        float("-inf")
                    )
                )

            if (
                top_p is not None
                and top_p < 1.0
            ):

                sorted_logits, sorted_indices = (
                    torch.sort(
                        next_token_logits,
                        descending=True,
                        dim=-1
                    )
                )

                sorted_probabilities = F.softmax(
                    sorted_logits,
                    dim=-1
                )

                cumulative_probabilities = (
                    torch.cumsum(
                        sorted_probabilities,
                        dim=-1
                    )
                )

                sorted_remove_mask = (
                    cumulative_probabilities
                    > top_p
                )

                sorted_remove_mask[
                    :,
                    1:
                ] = sorted_remove_mask[
                    :,
                    :-1
                ].clone()

                sorted_remove_mask[
                    :,
                    0
                ] = False

                remove_mask = torch.zeros_like(
                    sorted_remove_mask
                )

                remove_mask.scatter_(
                    dim=-1,
                    index=sorted_indices,
                    src=sorted_remove_mask
                )

                next_token_logits = (
                    next_token_logits.masked_fill(
                        remove_mask,
                        float("-inf")
                    )
                )

            probabilities = F.softmax(
                next_token_logits,
                dim=-1
            )

            if torch.isnan(
                probabilities
            ).any():

                raise RuntimeError(
                    "NaN values appeared in generation "
                    "probabilities."
                )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            generated_ids = torch.cat(
                [
                    generated_ids,
                    next_token
                ],
                dim=1
            )

            if end_token_id is not None:

                if torch.all(
                    next_token.squeeze(-1)
                    == end_token_id
                ):

                    break

        return generated_ids


# ============================================================
# 8. Parameter Counting
# ============================================================

def count_parameters(
    model: nn.Module,
    trainable_only: bool = True
) -> int:
    """
    Count model parameters.

    Parameters
    ----------
    model:
        PyTorch model.

    trainable_only:
        Count only parameters with requires_grad=True.
    """

    if trainable_only:

        return sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )

    return sum(
        parameter.numel()
        for parameter in model.parameters()
    )


# ============================================================
# 9. Tokenizer Loading
# ============================================================

def load_tokenizer(
    tokenizer_path: str
) -> Tokenizer:
    """
    Load a saved Hugging Face Tokenizers tokenizer.

    Parameters
    ----------
    tokenizer_path:
        Path to tokenizer.json.
    """

    if not isinstance(
        tokenizer_path,
        str
    ):

        raise TypeError(
            "tokenizer_path must be a string."
        )

    import os

    if not os.path.exists(
        tokenizer_path
    ):

        raise FileNotFoundError(
            "Tokenizer file was not found: "
            f"{tokenizer_path}"
        )

    tokenizer = Tokenizer.from_file(
        tokenizer_path
    )

    if tokenizer.get_vocab_size() <= 0:

        raise ValueError(
            "Loaded tokenizer has an empty vocabulary."
        )

    return tokenizer


# ============================================================
# 10. Prompt Creation
# ============================================================

def _clean_prompt_value(
    value: Any
) -> str:
    """
    Normalize a title or subject used in a prompt.
    """

    if value is None:
        return ""

    return " ".join(
        str(value).strip().split()
    )


def create_prompt(
    title: str,
    subject: str = "Machine Learning"
) -> str:
    """
    Create a structured prompt for abstract generation.

    Example
    -------
    <TITLE> Deep Learning for Medical Imaging
    <SUBJECT> Machine Learning
    <ABSTRACT>
    """

    cleaned_title = _clean_prompt_value(
        title
    )

    cleaned_subject = _clean_prompt_value(
        subject
    )

    if not cleaned_title:

        raise ValueError(
            "A research-paper title is required."
        )

    if not cleaned_subject:

        cleaned_subject = (
            "Machine Learning"
        )

    return (
        f"<TITLE> {cleaned_title} "
        f"<SUBJECT> {cleaned_subject} "
        f"<ABSTRACT>"
    )


# ============================================================
# 11. Text Generation Utility
# ============================================================

@torch.no_grad()
def generate_text(
    model: GPTLanguageModel,
    tokenizer: Tokenizer,
    prompt: str,
    device: Optional[
        Union[str, torch.device]
    ] = None,
    max_new_tokens: int = 250,
    temperature: float = 0.7,
    top_k: Optional[int] = 40,
    top_p: Optional[float] = None,
    repetition_penalty: float = 1.1,
    end_token: str = "<END>",
    skip_special_tokens: bool = False
) -> str:
    """
    Tokenize a prompt, generate tokens, and decode the result.
    """

    if not isinstance(
        model,
        GPTLanguageModel
    ):

        raise TypeError(
            "model must be a GPTLanguageModel instance."
        )

    if not isinstance(
        tokenizer,
        Tokenizer
    ):

        raise TypeError(
            "tokenizer must be a Tokenizer instance."
        )

    if not isinstance(
        prompt,
        str
    ) or not prompt.strip():

        raise ValueError(
            "prompt must be a non-empty string."
        )

    if device is None:

        device = next(
            model.parameters()
        ).device

    device = torch.device(
        device
    )

    prompt_encoding = tokenizer.encode(
        prompt
    )

    prompt_ids = prompt_encoding.ids

    if len(prompt_ids) == 0:

        raise ValueError(
            "The tokenizer produced an empty prompt."
        )

    prompt_tensor = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    end_token_id = tokenizer.token_to_id(
        end_token
    )

    model.eval()

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        end_token_id=end_token_id
    )

    decoded_text = tokenizer.decode(
        generated_ids[
            0
        ].detach().cpu().tolist(),
        skip_special_tokens=skip_special_tokens
    )

    return decoded_text


# ============================================================
# 12. Abstract Extraction
# ============================================================

def extract_abstract(
    generated_text: str,
    abstract_token: str = "<ABSTRACT>",
    end_token: str = "<END>"
) -> str:
    """
    Extract only the generated abstract from complete model text.
    """

    if generated_text is None:

        return ""

    text = str(
        generated_text
    )

    if abstract_token in text:

        text = text.split(
            abstract_token,
            1
        )[1]

    if end_token in text:

        text = text.split(
            end_token,
            1
        )[0]

    text = " ".join(
        text.strip().split()
    )

    return text


# ============================================================
# 13. Checkpoint Configuration Extraction
# ============================================================

def _extract_checkpoint_config(
    checkpoint: Dict[str, Any]
) -> GPTConfig:
    """
    Extract GPTConfig from a model checkpoint.
    """

    possible_config_keys = [
        "config",
        "model_config",
        "gpt_config"
    ]

    config_dictionary = None

    for key in possible_config_keys:

        if key in checkpoint:

            config_dictionary = (
                checkpoint[key]
            )

            break

    if config_dictionary is None:

        raise KeyError(
            "Checkpoint does not contain a model "
            "configuration. Expected one of: "
            f"{possible_config_keys}"
        )

    if isinstance(
        config_dictionary,
        GPTConfig
    ):

        return config_dictionary

    if not isinstance(
        config_dictionary,
        dict
    ):

        raise TypeError(
            "Checkpoint configuration must be "
            "a dictionary or GPTConfig."
        )

    return GPTConfig.from_dict(
        config_dictionary
    )


# ============================================================
# 14. Model Checkpoint Loading
# ============================================================

def load_model_checkpoint(
    checkpoint_path: str,
    device: Optional[
        Union[str, torch.device]
    ] = None,
    evaluation_mode: bool = True,
    strict: bool = True
) -> Tuple[
    GPTLanguageModel,
    GPTConfig,
    Dict[str, Any]
]:
    """
    Load a trained GPT model from a checkpoint.

    Supported checkpoint formats
    ----------------------------
    Recommended format:

    {
        "model_state_dict": model.state_dict(),
        "config": config.to_dict(),
        "step": step,
        "validation_loss": validation_loss
    }

    Also supports:

    {
        "state_dict": model.state_dict(),
        "config": ...
    }

    Returns
    -------
    model:
        Loaded GPTLanguageModel.

    config:
        GPTConfig used to create the model.

    checkpoint:
        Complete checkpoint dictionary.
    """

    import os

    if not os.path.exists(
        checkpoint_path
    ):

        raise FileNotFoundError(
            "Checkpoint was not found: "
            f"{checkpoint_path}"
        )

    if device is None:

        device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    else:

        device = torch.device(
            device
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    if not isinstance(
        checkpoint,
        dict
    ):

        raise TypeError(
            "Checkpoint must be a dictionary."
        )

    config = _extract_checkpoint_config(
        checkpoint
    )

    model = GPTLanguageModel(
        config
    ).to(device)

    if "model_state_dict" in checkpoint:

        model_state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        model_state_dict = checkpoint[
            "state_dict"
        ]

    else:

        raise KeyError(
            "Checkpoint does not contain "
            "'model_state_dict' or 'state_dict'."
        )

    model.load_state_dict(
        model_state_dict,
        strict=strict
    )

    if evaluation_mode:

        model.eval()

    return (
        model,
        config,
        checkpoint
    )
'''

with open(
    COMMON_MODULE_PATH,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        common_module_code
    )

print(
    "gpt_components.py created successfully:"
)

print(
    COMMON_MODULE_PATH
)

gpt_components.py created successfully:
/content/drive/MyDrive/Scientific-Abstract-GPT/src/gpt_components.py


Verify Created Files

In [7]:
created_files = {
    "__init__.py": INIT_FILE_PATH,
    "gpt_components.py": (
        COMMON_MODULE_PATH
    )
}

for file_name, file_path in (
    created_files.items()
):

    file_exists = os.path.exists(
        file_path
    )

    file_size = (
        os.path.getsize(
            file_path
        )
        if file_exists
        else 0
    )

    print(
        f"{file_name}: "
        f"{'FOUND' if file_exists else 'MISSING'}"
    )

    print(
        f"Size: {file_size:,} bytes"
    )

    print("-" * 60)

__init__.py: FOUND
Size: 677 bytes
------------------------------------------------------------
gpt_components.py: FOUND
Size: 32,848 bytes
------------------------------------------------------------


In [8]:
if PROJECT_PATH not in sys.path:

    sys.path.insert(
        0,
        PROJECT_PATH
    )

print(
    "Project added to Python path:"
)

print(
    PROJECT_PATH
)

Project added to Python path:
/content/drive/MyDrive/Scientific-Abstract-GPT


Import Common Components

In [9]:
from src.gpt_components import (
    GPTConfig,
    CausalSelfAttentionHead,
    MultiHeadAttention,
    FeedForward,
    TransformerBlock,
    GPTLanguageModel,
    set_seed,
    count_parameters,
    load_tokenizer,
    create_prompt,
    generate_text,
    extract_abstract,
    load_model_checkpoint
)

print(
    "Common GPT components "
    "imported successfully."
)

Common GPT components imported successfully.


In [10]:
set_seed(
    seed=42
)

print(
    "Random seed configured."
)

Random seed configured.


Test GPT Configuration

In [11]:
test_config = GPTConfig(
    vocab_size=8000,
    block_size=256,
    n_embd=256,
    n_head=8,
    n_layer=6,
    dropout=0.1,
    bias=True
)

print(test_config)

print(
    "\nHead size:",
    test_config.head_size
)

print(
    "\nConfiguration dictionary:"
)

print(
    json.dumps(
        test_config.to_dict(),
        indent=2
    )
)

GPTConfig(vocab_size=8000, block_size=256, n_embd=256, n_head=8, n_layer=6, dropout=0.1, bias=True)

Head size: 32

Configuration dictionary:
{
  "vocab_size": 8000,
  "block_size": 256,
  "n_embd": 256,
  "n_head": 8,
  "n_layer": 6,
  "dropout": 0.1,
  "bias": true
}


Create a Test GPT Model

In [12]:
test_model = GPTLanguageModel(
    test_config
)

print(test_model)

GPTLanguageModel(
  (token_embedding): Embedding(8000, 256)
  (position_embedding): Embedding(256, 256)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (layer_norm_attention): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (self_attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x CausalSelfAttentionHead(
            (key): Linear(in_features=256, out_features=32, bias=True)
            (query): Linear(in_features=256, out_features=32, bias=True)
            (value): Linear(in_features=256, out_features=32, bias=True)
            (attention_dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (output_projection): Linear(in_features=256, out_features=256, bias=True)
        (residual_dropout): Dropout(p=0.1, inplace=False)
      )
      (layer_norm_feed_forward): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (feed_forward): FeedForward(
     

In [13]:
trainable_parameter_count = (
    count_parameters(
        test_model,
        trainable_only=True
    )
)

total_parameter_count = (
    count_parameters(
        test_model,
        trainable_only=False
    )
)

print(
    "Trainable parameters:",
    f"{trainable_parameter_count:,}"
)

print(
    "Total parameters:",
    f"{total_parameter_count:,}"
)

Trainable parameters: 6,852,608
Total parameters: 6,852,608


Test Forward Pass

In [14]:
import torch

batch_size = 4
sequence_length = 64

test_inputs = torch.randint(
    low=0,
    high=test_config.vocab_size,
    size=(
        batch_size,
        sequence_length
    ),
    dtype=torch.long
)

test_targets = torch.randint(
    low=0,
    high=test_config.vocab_size,
    size=(
        batch_size,
        sequence_length
    ),
    dtype=torch.long
)

test_logits, test_loss = test_model(
    test_inputs,
    test_targets
)

print(
    "Input shape:",
    test_inputs.shape
)

print(
    "Logits shape:",
    test_logits.shape
)

print(
    "Loss:",
    test_loss.item()
)

Input shape: torch.Size([4, 64])
Logits shape: torch.Size([4, 64, 8000])
Loss: 9.029844284057617


Verify Forward-Pass Shapes

In [15]:
expected_logits_shape = (
    batch_size,
    sequence_length,
    test_config.vocab_size
)

if test_logits.shape != expected_logits_shape:

    raise ValueError(
        "Incorrect logits shape. "
        f"Expected {expected_logits_shape}, "
        f"received {tuple(test_logits.shape)}."
    )

if test_loss.ndim != 0:

    raise ValueError(
        "Loss should be a scalar tensor."
    )

print(
    "Forward-pass validation passed."
)

Forward-pass validation passed.


Test Model Generation

This is only a model-level generation test using random token IDs. It is not expected to produce readable text because the test model is untrained.

In [16]:
generation_prompt = torch.tensor(
    [
        [2, 10, 20, 30]
    ],
    dtype=torch.long
)

generated_token_ids = test_model.generate(
    input_ids=generation_prompt,
    max_new_tokens=20,
    temperature=0.8,
    top_k=40,
    repetition_penalty=1.1
)

print(
    "Prompt shape:",
    generation_prompt.shape
)

print(
    "Generated shape:",
    generated_token_ids.shape
)

print(
    "Generated token IDs:"
)

print(
    generated_token_ids
)

Prompt shape: torch.Size([1, 4])
Generated shape: torch.Size([1, 24])
Generated token IDs:
tensor([[   2,   10,   20,   30, 3351, 7650, 1130, 5832, 5782, 3747, 5235, 4413,
         7735, 1353, 2542, 4675, 2628, 1177, 1177, 7193, 7886, 7740,  281, 6545]])


Test Prompt Creation

In [17]:
test_prompt = create_prompt(
    title=(
        "Deep Learning for "
        "Medical Image Classification"
    ),
    subject="Machine Learning"
)

print(test_prompt)

<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT>


Test Abstract Extraction

In [18]:
test_complete_output = (
    "<TITLE> Deep Learning for Medical Imaging "
    "<SUBJECT> Machine Learning "
    "<ABSTRACT> This paper proposes a new "
    "deep learning method for medical-image "
    "classification. <END>"
)

test_extracted_abstract = (
    extract_abstract(
        test_complete_output
    )
)

print(
    "Complete output:"
)

print(
    test_complete_output
)

print(
    "\nExtracted abstract:"
)

print(
    test_extracted_abstract
)

Complete output:
<TITLE> Deep Learning for Medical Imaging <SUBJECT> Machine Learning <ABSTRACT> This paper proposes a new deep learning method for medical-image classification. <END>

Extracted abstract:
This paper proposes a new deep learning method for medical-image classification.


Test Real BPE Tokenizer Loading


In [19]:
TOKENIZER_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "bpe_tokenizer",
    "tokenizer.json"
)

if os.path.exists(
    TOKENIZER_PATH
):

    test_tokenizer = load_tokenizer(
        TOKENIZER_PATH
    )

    print(
        "Tokenizer vocabulary size:",
        test_tokenizer.get_vocab_size()
    )

    tokenizer_test_encoding = (
        test_tokenizer.encode(
            test_prompt
        )
    )

    print(
        "\nPrompt token IDs:"
    )

    print(
        tokenizer_test_encoding.ids
    )

    print(
        "\nPrompt tokens:"
    )

    print(
        tokenizer_test_encoding.tokens
    )

else:

    print(
        "Tokenizer has not been created yet."
    )

    print(
        "Run 02_Tokenization.ipynb before "
        "performing the tokenizer test."
    )

Tokenizer vocabulary size: 8000

Prompt token IDs:
[2, 1290, 468, 324, 5478, 3843, 2706, 226, 3, 536, 468, 226, 4]

Prompt tokens:
['<TITLE>', 'ĠDeep', 'ĠLearning', 'Ġfor', 'ĠMedical', 'ĠImage', 'ĠClassification', 'Ġ', '<SUBJECT>', 'ĠMachine', 'ĠLearning', 'Ġ', '<ABSTRACT>']


In [20]:
if os.path.exists(
    TOKENIZER_PATH
):

    actual_vocab_size = (
        test_tokenizer.get_vocab_size()
    )

    real_config = GPTConfig(
        vocab_size=actual_vocab_size,
        block_size=256,
        n_embd=256,
        n_head=8,
        n_layer=6,
        dropout=0.1,
        bias=True
    )

    print(
        "Real tokenizer vocabulary size:",
        actual_vocab_size
    )

    print(
        "\nRecommended model configuration:"
    )

    print(
        json.dumps(
            real_config.to_dict(),
            indent=2
        )
    )

else:

    print(
        "Real model configuration test skipped."
    )

Real tokenizer vocabulary size: 8000

Recommended model configuration:
{
  "vocab_size": 8000,
  "block_size": 256,
  "n_embd": 256,
  "n_head": 8,
  "n_layer": 6,
  "dropout": 0.1,
  "bias": true
}


In [21]:
import py_compile

py_compile.compile(
    COMMON_MODULE_PATH,
    doraise=True
)

print(
    "Python syntax validation passed."
)

Python syntax validation passed.


Reload Module Test

This confirms that the module can be imported after a Colab runtime reload.

In [22]:
import src.gpt_components as gpt_components

importlib.reload(
    gpt_components
)

print(
    "Module reloaded successfully."
)

print(
    "Available configuration class:"
)

print(
    gpt_components.GPTConfig
)

print(
    "\nAvailable model class:"
)

print(
    gpt_components.GPTLanguageModel
)

Module reloaded successfully.
Available configuration class:
<class 'src.gpt_components.GPTConfig'>

Available model class:
<class 'src.gpt_components.GPTLanguageModel'>


Final Validation

In [23]:
validation_checks = {
    "source_folder_exists": (
        os.path.exists(
            SRC_FOLDER
        )
    ),
    "init_file_exists": (
        os.path.exists(
            INIT_FILE_PATH
        )
    ),
    "common_module_exists": (
        os.path.exists(
            COMMON_MODULE_PATH
        )
    ),
    "forward_pass_completed": (
        test_logits is not None
    ),
    "loss_calculated": (
        test_loss is not None
    ),
    "logits_shape_valid": (
        tuple(
            test_logits.shape
        )
        == expected_logits_shape
    ),
    "generation_completed": (
        generated_token_ids.shape[1]
        == generation_prompt.shape[1] + 20
    ),
    "prompt_created": (
        test_prompt.startswith(
            "<TITLE>"
        )
    ),
    "abstract_extracted": (
        test_extracted_abstract.startswith(
            "This paper"
        )
    )
}

for check_name, result in (
    validation_checks.items()
):

    status = (
        "PASSED"
        if result
        else "FAILED"
    )

    print(
        f"{check_name}: {status}"
    )

source_folder_exists: PASSED
init_file_exists: PASSED
common_module_exists: PASSED
forward_pass_completed: PASSED
loss_calculated: PASSED
logits_shape_valid: PASSED
generation_completed: PASSED
prompt_created: PASSED
abstract_extracted: PASSED


In [24]:
if all(
    validation_checks.values()
):

    print("=" * 80)

    print(
        "00_Common_GPT_Components.ipynb "
        "completed successfully."
    )

    print("=" * 80)

    print(
        "\nReusable module created:"
    )

    print(
        COMMON_MODULE_PATH
    )

    print(
        "\nThe following notebooks can now "
        "import the common components:"
    )

    print(
        "03_GPT_Model.ipynb"
    )

    print(
        "04_Training.ipynb"
    )

    print(
        "05_Text_Generation.ipynb"
    )

    print(
        "06_Evaluation.ipynb"
    )

    print(
        "07_Demo.ipynb"
    )

else:

    failed_checks = [
        check_name
        for check_name, result
        in validation_checks.items()
        if not result
    ]

    raise RuntimeError(
        "Common-component validation failed: "
        f"{failed_checks}"
    )

00_Common_GPT_Components.ipynb completed successfully.

Reusable module created:
/content/drive/MyDrive/Scientific-Abstract-GPT/src/gpt_components.py

The following notebooks can now import the common components:
03_GPT_Model.ipynb
04_Training.ipynb
05_Text_Generation.ipynb
06_Evaluation.ipynb
07_Demo.ipynb
